In [1]:
# The purpose of this notebook is that you need to join together all the files, 
# and then normalise them, and get them into a format that matches the format that Clara had for them 

In [2]:
import numpy as np
import xarray as xr    
import glob
import pandas as pd
import itertools
import sklearn

In [3]:
# Set all the filepaths that will be required 
filepath_data_ho = "/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/"
filepath_masks = filepath_data_ho + "masks_"
#filepath_nico_on_nemo = filepath_data_ho + "nico_on_nemo_" + nemo_run + '.nc'
#filepath_mask_nemo_run = filepath_masks + nemo_run + '.nc'
filepath_nn_input = filepath_data_ho + "nn_input_"
#filepath_slopes = filepath_nn_input + nemo_run + '_' + 'redo_slopes2' + '.nc'
filepath_areas = "/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Masks/areas.nc"

# Set where you would like to save the data 
data_out_fp = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/'
# Create a dataframe with all the data which is saved in this location
#intermediate_filepath = data_out_fp + nemo_run + '_' + 'whole_dataset' + '_' + 'not_yet_normalised.csv'
# Creat a dataframe with only the specified data (which is also saved in this location) 
#this_collection = 'no_mar_oct'
#fp_metrics = data_out_fp + this_collection + '_' + 'metrics_norm.nc'
#fp_var_train_norm = data_out_fp + this_collection + '_' + 'train_data.nc'
#fp_var_val_norm = data_out_fp + this_collection + '_' + 'val_data.nc'

### **You can start here if you've already merged together the simulations into a pandas dataframe (saved as intermediate_filepath)**

**Separate out the testing data, and then split the rest into training and validation datasets**

In [34]:
this_collection = 'OPM021'#_OPM0263_Christoph'

In [35]:
collections = this_collection.split('_')
print(collections)

['OPM021']


In [36]:
filepaths = []
for i in range(len(collections)):
    if collections[i] == 'Christoph':
        collections[i] = 'Christoph_v2'
    filepaths.append(data_out_fp + collections[i] + '_' + 'whole_dataset' + '_' + 'not_yet_normalised.csv')

In [37]:
fast_check_version = False
if fast_check_version == True:
    df_total = pd.read_csv(filepaths[0], nrows = 5)
    print('You have loaded', filepaths[0])
    for i in range(len(filepaths)-1):
        df_total = pd.concat([df_total, pd.read_csv(filepaths[i+1], nrows = 5)])
        print('You have loaded', filepaths[i+1])
    print('There are', df_total.shape[0], 'entries in the combined dataset')
elif fast_check_version == False:
    df_total = pd.read_csv(filepaths[0])
    print('You have loaded', filepaths[0])
    for i in range(len(filepaths)-1):
        df_total = pd.concat([df_total, pd.read_csv(filepaths[i+1])])
        print('You have loaded', filepaths[i+1])
    print('There are', df_total.shape[0], 'entries in the combined dataset')

You have loaded /bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Training_data/OPM021_whole_dataset_not_yet_normalised.csv
There are 340140 entries in the combined dataset


In [40]:
2018-1989

29

In [39]:
np.unique(df_total.year)

array([1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999,
       2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018])

In [18]:
np.sum(np.isnan(df_total))

lat                            0
lon                            0
temperature_prop               0
salinity_prop                  0
melt_m_ice_per_y               0
mean_T                    290400
mean_S                    290400
std_T                     290400
std_S                     290400
year                           0
month                          0
basins_NEMO                    0
distances_GL                   0
distances_OO                   0
distances_OC                   0
corrected_isdraft              0
area                           0
bathymetry                     0
slope_is_lon                   0
slope_is_lat                   0
slope_ba_lon                   0
slope_ba_lat                   0
slope_is_across_front          0
slope_is_towards_front         0
slope_ba_across_front          0
slope_ba_towards_front         0
dtype: int64

In [13]:
annual_train = False
if annual_train == True:
    df_grouped = df_total.groupby(['lat','lon','year'], as_index = False).mean()
    df_total = df_grouped
    annual_f = 'annual_'
else:   
    annual_f = ''

In [17]:
# Split the training and validation datasets 
fraction_for_validation = 1/9 # How much of the training/validation dataset to use for validation
                              # Assuming a 80:10:10 training:validation:testing split, set this to 1/9 
                              #    (as 1/10 for testing has already been taken)
                              # You can set this to False to use the whole dataset for training and validation
if fraction_for_validation == False:
    train_input_df1 = df_total.copy()
    val_input_df1 = df_total.copy()
    no_train = df_total.shape[0]
    no_val = df_total.shape[0]
    print('Warning: You are not creating a separate validation dataset, this may affect the robustmess of your results')
else:
    # Split the data with the desired ratio (it is also shuffled)
    train_input_df1, val_input_df1 = \
            sklearn.model_selection.train_test_split(df_total, test_size = fraction_for_validation, random_state = 1)
    no_train = train_input_df1.shape[0]
    no_val = val_input_df1.shape[0]
no_test = df_total.shape[0] - no_train - no_val
print('Training data:   {} points, {:.0f}% of data'.format(no_train, no_train*100/df_total.shape[0]))
print('Validation data: {} points,  {:.0f}% of data'.format(no_val, no_val*100/df_total.shape[0]))
print('Testing data:    {} points,  {:.0f}% of data'.format(no_test, no_test*100/df_total.shape[0]))

train_input_df = train_input_df1.to_xarray()
val_input_df = val_input_df1.to_xarray()

## Prepare the training and validation datasets
y_train = train_input_df['melt_m_ice_per_y']
x_train = train_input_df.drop_vars(['melt_m_ice_per_y'])
y_val = val_input_df['melt_m_ice_per_y']
x_val = val_input_df.drop_vars(['melt_m_ice_per_y'])
print()
print('Training and validation datasets (x and y) prepared')

Training data:   29942293 points, 89% of data
Validation data: 3742787 points,  11% of data
Testing data:    0 points,  0% of data

Training and validation datasets (x and y) prepared


**Normalise the dataset**

In [20]:
def compute_norm_metrics(x_train, y_train, norm_method):
    # Calculate the mean
    x_mean = x_train.mean()
    y_mean = y_train.mean()
    # Calulate the normalisation factor 
    if norm_method == 'std':
        x_range  = x_train.std()
        y_range  = y_train.std()
    elif norm_method == 'interquart':
        x_range  = x_train.quantile(0.9) - x_train.quantile(0.1)
        y_range  = y_train.quantile(0.9) - y_train.quantile(0.1)
    elif norm_method == 'minmax':
        x_range  = x_train.max() - x_train.min() 
        y_range  = y_train.max() - y_train.min() 
    # Merge methods 
    norm_mean = xr.merge([x_mean,y_mean]).assign_coords({'metric': 'mean_vars', 'norm_method': norm_method})
    norm_range = xr.merge([x_range,y_range]).assign_coords({'metric': 'range_vars', 'norm_method': norm_method})
    # Create array of metrics
    summary_metrics = xr.concat([norm_mean, norm_range], dim='metric').assign_coords({'norm_method': norm_method})
    return summary_metrics

In [21]:
print(this_collection)


OPM026_OPM031_OPM0263


In [ ]:
# Normalise the input and output data
norm_summary_list = []
for norm_method in ['std','interquart','minmax']:
    summary_ds = compute_norm_metrics(x_train, y_train, norm_method)
    norm_summary_list.append(summary_ds)
summary_ds_all = xr.concat(norm_summary_list, dim='norm_method')
print('Data normalised for all three methods')

# Calculate var mean, var 
var_mean = summary_ds_all.sel(metric='mean_vars')
var_range = summary_ds_all.sel(metric='range_vars')
var_train_norm = (train_input_df - var_mean)/var_range
var_val_norm = (val_input_df - var_mean)/var_range
print('Normalisation metrics calculated')

#set filenames
fp_metrics = data_out_fp + this_collection + '_' + annual_f + 'metrics_norm.nc'
fp_var_train_norm = data_out_fp + this_collection + '_' + annual_f + 'train_data.nc'
fp_var_val_norm = data_out_fp + this_collection + '_' + annual_f + 'val_data.nc'
#print(fp_metrics)
#print(fp_var_train_norm)
#print(fp_var_val_norm)

# Set data to variables and save to file
metrics_ds, var_train_norm, var_val_norm = summary_ds_all, var_train_norm, var_val_norm
metrics_ds.to_netcdf(fp_metrics)
var_train_norm.to_netcdf(fp_var_train_norm)
var_val_norm.to_netcdf(fp_var_val_norm)    
print('You have saved:')
print(fp_metrics)
print(fp_var_train_norm)
print(fp_var_val_norm)

Data normalised for all three methods


In [10]:
annual_f = ''

In [11]:
fp_metrics = data_out_fp + this_collection + '_' + annual_f + 'metrics_norm.nc'

In [12]:
xr.load_dataset(fp_metrics)

<xarray.Dataset>
Dimensions:                 (norm_method: 3, metric: 2)
Coordinates:
  * metric                  (metric) object 'mean_vars' 'range_vars'
  * norm_method             (norm_method) object 'std' 'interquart' 'minmax'
Data variables: (12/26)
    lat                     (norm_method, metric) float64 -77.74 4.81 ... 20.4
    lon                     (norm_method, metric) float64 -55.3 80.23 ... 359.8
    temperature_prop        (norm_method, metric) float64 -1.905 0.503 ... 4.501
    salinity_prop           (norm_method, metric) float64 34.63 0.2327 ... 5.275
    mean_T                  (norm_method, metric) float64 -1.905 ... 4.017
    mean_S                  (norm_method, metric) float64 34.63 0.1983 ... 3.287
    ...                      ...
    slope_ba_lat            (norm_method, metric) float64 0.0002204 ... 0.2353
    slope_is_across_front   (norm_method, metric) float64 0.0001662 ... 0.1376
    slope_is_towards_front  (norm_method, metric) float64 0.0002593 ... 0.113
    slope_ba_across_front   (norm_method, metric) float64 -0.0003064 ... 0.1498
    slope_ba_towards_front  (norm_method, metric) float64 0.0009622 ... 0.2302
    melt_m_ice_per_y        (norm_method, metric) float64 -3.185e-05 ... 0.00...